In [30]:
# CELL 1 — Setup (run every session, no restart needed)
import os, sys, subprocess, json as _json

# ── bitsandbytes env vars ─────────────────────────────────────
os.environ["BNB_CUDA_VERSION"] = "128"
os.environ["CUDA_VERSION"]     = "128"
os.environ["LD_LIBRARY_PATH"]  = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH","")

def pip(cmd):
    subprocess.run(f"pip install --quiet {cmd}", shell=True, check=False)

# ── Install missing packages ──────────────────────────────────
pip("peft>=0.12.0 accelerate>=0.30.0")
pip("sentence-transformers>=2.7.0")
pip("huggingface_hub>=0.23.0 datasets>=2.14.0")

# ── Patch transformers/utils/__init__.py on disk ──────────────
# Newer Kaggle transformers moved is_flax_available out of utils/__init__.py
# but clip/__init__.py still imports it from there.
# We add it back by patching the file directly.

import glob
utils_init = "/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py"

if os.path.exists(utils_init):
    with open(utils_init) as f:
        content = f.read()

    missing = []
    needs = [
        "is_flax_available",
        "is_tf_available",
        "is_torch_tpu_available",
    ]
    for fn in needs:
        if f"def {fn}" not in content and f"is_flax_available" not in content[:500]:
            missing.append(fn)

    if missing or "is_flax_available" not in content:
        patch = """
# --- compatibility patch ---
try:
    from transformers.utils.import_utils import (
        is_flax_available,
        is_tf_available,
        is_torch_tpu_available,
    )
except ImportError:
    def is_flax_available(): return False
    def is_tf_available(): return False
    def is_torch_tpu_available(): return False
# --- end patch ---
"""
        if "compatibility patch" not in content:
            with open(utils_init, "a") as f:
                f.write(patch)
            print(f"Patched {utils_init}")
        else:
            print(f"Already patched {utils_init}")
    else:
        print(f"No patch needed for {utils_init}")
else:
    print(f"File not found: {utils_init}")

# ── Patch cached preprocessor_config.json ────────────────────
CACHE_DIR = "/kaggle/working/llava_cache"
patched = 0
if os.path.exists(CACHE_DIR):
    for root, dirs, files in os.walk(CACHE_DIR):
        for fname in files:
            if fname == "preprocessor_config.json":
                fpath = os.path.join(root, fname)
                try:
                    with open(fpath) as f:
                        cfg = _json.load(f)
                    if "auto_map" in cfg:
                        del cfg["auto_map"]
                        with open(fpath, "w") as f:
                            _json.dump(cfg, f, indent=2)
                        patched += 1
                        print(f"Patched cache: {fpath}")
                except Exception as e:
                    print(f"Could not patch {fpath}: {e}")
if patched == 0 and os.path.exists(CACHE_DIR):
    print("No cache files needed patching.")

# ── Verify ────────────────────────────────────────────────────
print("\nVerifying...")
# Force reload transformers after patch
import importlib
if "transformers" in sys.modules:
    # Clear the cached broken modules
    to_remove = [k for k in sys.modules if k.startswith("transformers")]
    for k in to_remove:
        del sys.modules[k]
    print(f"  Cleared {len(to_remove)} cached transformers modules")

try:
    import transformers
    from transformers import LlavaForConditionalGeneration
    print(f"  OK  transformers  {transformers.__version__}")
    print(f"  OK  LlavaForConditionalGeneration")
except Exception as e:
    print(f"  ERR transformers: {e}")

try:
    import peft
    print(f"  OK  peft  {peft.__version__}")
except Exception as e:
    print(f"  ERR peft: {e}")

print("\nDone. Proceed to Cell 2.")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


Patched /usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py
No cache files needed patching.

Verifying...
  Cleared 200 cached transformers modules


0it [00:00, ?it/s]

  OK  transformers  4.44.2
  OK  LlavaForConditionalGeneration
  OK  peft  0.18.1

Done. Proceed to Cell 2.


In [31]:
# CELL 2 — Imports
import os, gc, json, warnings, random, sys

# ── bitsandbytes env vars ─────────────────────────────────────
os.environ["BNB_CUDA_VERSION"] = "128"
os.environ["CUDA_VERSION"]     = "128"
os.environ["LD_LIBRARY_PATH"]  = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH","")

# ── Clear stale transformers cache so patched file is read ────
stale = [k for k in sys.modules if k.startswith("transformers")]
for k in stale:
    del sys.modules[k]

# ── Imports ───────────────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from scipy import stats as scipy_stats
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    AutoTokenizer,
    CLIPImageProcessor,
    LlavaProcessor,
)
from peft import LoraConfig, get_peft_model
from huggingface_hub import snapshot_download

warnings.filterwarnings("ignore")

import transformers, peft
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"CUDA         {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

try:
    import bitsandbytes as bnb
    print(f"bitsandbytes {bnb.__version__}  OK")
    BNB_OK = True
except Exception as e:
    print(f"bitsandbytes unavailable — using float16 fallback")
    BNB_OK = False

print("All imports OK")


2026-06-10 02:00:29.128011: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781056829.326150      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781056829.391264      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781056829.881330      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781056829.881367      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781056829.881370      58 computation_placer.cc:177] computation placer alr

torch        2.10.0+cu128
transformers 4.44.2
peft         0.18.1
CUDA         True | Tesla P100-PCIE-16GB
bitsandbytes unavailable — using float16 fallback
All imports OK


In [32]:
# CELL 3 — Configuration
import os

WORK_DIR  = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
CACHE_DIR = os.path.join(WORK_DIR, "llava_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

CFG = {
    # Model
    "MODEL_ID":            "llava-hf/llava-1.5-7b-hf",
    "LORA_RANK":           16,
    # Training
    "PPO_STEPS":           400,
    "TRAIN_ITEMS":         400,
    "LEARNING_RATE":       1e-5,
    "TEMPERATURE":         0.9,
    "MAX_NEW_TOKENS":      50,
    "EMA_DECAY":           0.9,
    "GRAD_CLIP":           0.5,
    # PPO-Clip
    "PPO_EPSILON":         0.2,
    # Evaluation
    "TEST_ITEMS":          500,
    "EVAL_TEMPERATURE":    0.7,
    # Seeds (8 seeds)
    "SEEDS":               [42, 123, 7, 1, 2, 3, 4, 5],
    # Reward
    "W_TASK":              1.0,
    "W_FACT":              0.5,
    "PENALTY":            -0.5,
    "FGM_THRESHOLD":       0.6,
    "TASK_THRESHOLD":      0.7,
    # Stats
    "CI_LEVEL":            0.95,
}

print("CFG ready:")
for k,v in CFG.items():
    print(f"  {k:<25} {v}")


CFG ready:
  MODEL_ID                  llava-hf/llava-1.5-7b-hf
  LORA_RANK                 16
  PPO_STEPS                 400
  TRAIN_ITEMS               400
  LEARNING_RATE             1e-05
  TEMPERATURE               0.9
  MAX_NEW_TOKENS            50
  EMA_DECAY                 0.9
  GRAD_CLIP                 0.5
  PPO_EPSILON               0.2
  TEST_ITEMS                500
  EVAL_TEMPERATURE          0.7
  SEEDS                     [42, 123, 7, 1, 2, 3, 4, 5]
  W_TASK                    1.0
  W_FACT                    0.5
  PENALTY                   -0.5
  FGM_THRESHOLD             0.6
  TASK_THRESHOLD            0.7
  CI_LEVEL                  0.95


In [33]:
# CELL 4 — Environment Setup
# Downloads model weights once, sets up random seed utility

from huggingface_hub import snapshot_download

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def nuke(*extra_names):
    """Free all known model variables from GPU memory."""
    names = list(extra_names) + [
        "model","base","sm_reinforce","sm_ppoclip","sm_hra",
        "vm","vanilla_model","lora_model"
    ]
    for name in names:
        if name in globals():
            try:    globals()[name].cpu()
            except: pass
            try:    del globals()[name]
            except: pass
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total")

# Download weights once (file transfer only — no GPU)
already_cached = any(
    f.endswith(".safetensors")
    for _, _, files in os.walk(CACHE_DIR)
    for f in files
)

if already_cached:
    print(f"✓ Model weights cached at {CACHE_DIR}")
else:
    print(f"Downloading model weights to {CACHE_DIR} ...")
    print("  (One-time download ~14GB — subsequent runs load from disk)")
    snapshot_download(
        repo_id=CFG["MODEL_ID"],
        cache_dir=CACHE_DIR,
        ignore_patterns=["*.msgpack","*.h5","flax_model*","tf_model*"],
    )
    already_cached = True
    print("✓ Download complete")

print(f"\nWorking dir : {WORK_DIR}")
print(f"Cache dir   : {CACHE_DIR}")
print(f"Cached      : {already_cached}")


✓ Model weights cached at /kaggle/working/llava_cache

Working dir : /kaggle/working
Cache dir   : /kaggle/working/llava_cache
Cached      : True


In [34]:
# CELL 5 — Model Factory

def _bnb():
    if not BNB_OK:
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
        bnb_4bit_quant_type="nf4",
    )

def _lora():
    return LoraConfig(
        r=CFG["LORA_RANK"],
        lora_alpha=CFG["LORA_RANK"] * 2,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj","v_proj"],
        task_type="CAUSAL_LM",
    )

def make_lora_model():
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM before load: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")

    bnb_cfg = _bnb()
    kwargs = dict(
        device_map="auto",
        cache_dir=CACHE_DIR,
        local_files_only=already_cached,
    )
    if bnb_cfg is not None:
        kwargs["quantization_config"] = bnb_cfg
        print("  Loading in 4-bit (bitsandbytes)")
    else:
        kwargs["torch_dtype"] = torch.float16
        print("  Loading in float16 (bitsandbytes unavailable)")

    base  = LlavaForConditionalGeneration.from_pretrained(CFG["MODEL_ID"], **kwargs)
    model = get_peft_model(base, _lora())
    model.print_trainable_parameters()
    free2, _ = torch.cuda.mem_get_info()
    print(f"  VRAM after load:  {free2/1024**3:.2f} GB")
    return model

def make_vanilla_model():
    gc.collect(); torch.cuda.empty_cache()
    bnb_cfg = _bnb()
    kwargs = dict(
        device_map="auto",
        cache_dir=CACHE_DIR,
        local_files_only=already_cached,
    )
    if bnb_cfg is not None:
        kwargs["quantization_config"] = bnb_cfg
    else:
        kwargs["torch_dtype"] = torch.float16

    m = LlavaForConditionalGeneration.from_pretrained(CFG["MODEL_ID"], **kwargs)
    free, _ = torch.cuda.mem_get_info()
    print(f"  Vanilla loaded. VRAM: {free/1024**3:.2f} GB")
    return m

print("Model factory ready.")
print(f"  Quantisation: {'4-bit (bitsandbytes)' if BNB_OK else 'float16 fallback'}")


Model factory ready.
  Quantisation: float16 fallback


In [36]:
# CELL 6 — Processor and FGM

# ── Processor ────────────────────────────────────────────────
image_processor = CLIPImageProcessor.from_pretrained(
    CFG["MODEL_ID"],
    cache_dir=CACHE_DIR,
    local_files_only=already_cached,
)
tokenizer = AutoTokenizer.from_pretrained(
    CFG["MODEL_ID"],
    cache_dir=CACHE_DIR,
    local_files_only=already_cached,
)
processor = LlavaProcessor(
    image_processor=image_processor,
    tokenizer=tokenizer,
)
processor.tokenizer.pad_token    = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"
print(f"Processor ready  (vocab: {processor.tokenizer.vocab_size})")

# ── FGM: implemented directly with transformers ───────────────
# Avoids sentence_transformers entirely (version conflicts on Kaggle)
from transformers import AutoTokenizer as _FGMTokenizer, AutoModel as _FGMModel

_fgm_tok = _FGMTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
_fgm_mod = _FGMModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
_fgm_mod.eval()

def fgm_score(response: str, ground_truth: str) -> float:
    """Cosine similarity between mean-pooled BERT embeddings."""
    enc = _fgm_tok(
        [response, ground_truth],
        padding=True, truncation=True,
        max_length=128, return_tensors="pt"
    )
    with torch.no_grad():
        out = _fgm_mod(**enc)
    # Mean pooling
    tok_emb = out.last_hidden_state            # (2, seq, 384)
    mask    = enc["attention_mask"].unsqueeze(-1).float()
    emb     = (tok_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    emb     = torch.nn.functional.normalize(emb, p=2, dim=1)
    return float(torch.nn.functional.cosine_similarity(
        emb[0:1], emb[1:2]
    ).item())

# Quick self-test
score = fgm_score("the cat sat on the mat", "the cat sat on the mat")
print(f"FGM ready        (all-MiniLM-L6-v2, direct transformers)")
print(f"FGM self-test:   {score:.4f}  (expect ~1.0)")


Processor ready  (vocab: 32000)


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

FGM ready        (all-MiniLM-L6-v2, direct transformers)
FGM self-test:   1.0000  (expect ~1.0)


In [37]:
# CELL 7 — Reward Function + Ablation Configs

REWARD_CONFIGS = {
    "A: Task Only":    {"w_task":1.0, "w_fact":0.0, "penalty":0.0},
    "B: Fact Only":    {"w_task":0.0, "w_fact":1.0, "penalty":-0.5},
    "C: No Penalty":   {"w_task":1.0, "w_fact":0.5, "penalty":0.0},
    "D: Full HRA":     {"w_task":1.0, "w_fact":0.5, "penalty":-0.5},
    "E: Std REINFORCE":{"w_task":1.0, "w_fact":0.0, "penalty":0.0},
}

def compute_reward(config_name: str, fact_score: float) -> float:
    cfg = REWARD_CONFIGS.get(config_name, REWARD_CONFIGS["D: Full HRA"])
    r_task = 1.0 if fact_score > CFG["TASK_THRESHOLD"] else 0.0
    r_fact = fact_score if fact_score >= CFG["FGM_THRESHOLD"] else cfg["penalty"]
    return cfg["w_task"] * r_task + cfg["w_fact"] * r_fact

print("✓ Reward configs:")
for name, cfg in REWARD_CONFIGS.items():
    print(f"  {name}: {cfg}")

✓ Reward configs:
  A: Task Only: {'w_task': 1.0, 'w_fact': 0.0, 'penalty': 0.0}
  B: Fact Only: {'w_task': 0.0, 'w_fact': 1.0, 'penalty': -0.5}
  C: No Penalty: {'w_task': 1.0, 'w_fact': 0.5, 'penalty': 0.0}
  D: Full HRA: {'w_task': 1.0, 'w_fact': 0.5, 'penalty': -0.5}
  E: Std REINFORCE: {'w_task': 1.0, 'w_fact': 0.0, 'penalty': 0.0}


In [38]:
# CELL 8 — Datasets (ScienceQA + VQA-v2)

from datasets import load_dataset

# ── ScienceQA ─────────────────────────────────────────────
print("Loading ScienceQA...")
_scienceqa = {
    "train": load_dataset("derek-thomas/ScienceQA", split="train",  trust_remote_code=True),
    "test":  load_dataset("derek-thomas/ScienceQA", split="test",   trust_remote_code=True),
}
print(f"  train: {len(_scienceqa['train'])}  test: {len(_scienceqa['test'])}")

def build_scienceqa(split="train", max_items=400, seed=42):
    ds = _scienceqa[split]
    indices = [i for i,r in enumerate(ds) if r.get("image") is not None]
    random.seed(seed); random.shuffle(indices)
    items = []
    for i in indices[:max_items]:
        row = ds[i]
        try:
            img = row["image"]
            if not isinstance(img, Image.Image): continue
            if img.mode != "RGB": img = img.convert("RGB")
            choices  = row.get("choices", [])
            answer   = row.get("answer", 0)
            gt       = choices[answer] if choices and isinstance(answer,int) and answer < len(choices) else str(answer)
            question = row.get("question","")
            if choices:
                question += " Choices: " + ", ".join(choices)
            items.append({"prompt":question,"ground_truth":gt,"image":img,"source":"scienceqa"})
        except: continue
    print(f"  ScienceQA {split}: {len(items)} items (seed={seed})")
    return items

# ── VQA-v2 (streaming) ────────────────────────────────────
def build_vqa(max_items=500, skip=0, seed=42):
    stream = load_dataset("lmms-lab/VQAv2", split="validation", streaming=True)
    items, skipped = [], 0
    for row in stream:
        if skipped < skip: skipped += 1; continue
        if len(items) >= max_items: break
        try:
            raw = row.get("answers",[])
            if not raw: continue
            answers = [a["answer"] for a in raw] if isinstance(raw[0],dict) else list(raw)
            gt = max(set(answers), key=answers.count)
            img = row.get("image") or row.get("img")
            if img is None or not isinstance(img, Image.Image): continue
            if img.mode != "RGB": img = img.convert("RGB")
            items.append({"prompt":row.get("question",""),"ground_truth":gt,
                          "image":img,"source":"vqa_v2"})
        except: continue
    print(f"  VQA-v2: {len(items)} items (skip={skip})")
    return items

# Pre-build fixed test sets (used across all seeds)
print("\nBuilding fixed test sets...")
SCIQA_TEST = build_scienceqa("test",  max_items=CFG["TEST_ITEMS"],  seed=42)
VQA_TEST   = build_vqa(max_items=CFG["TEST_ITEMS"], skip=0, seed=42)

print(f"\n✓ Test sets ready:")
print(f"  ScienceQA test : {len(SCIQA_TEST)} items")
print(f"  VQA-v2 test    : {len(VQA_TEST)} items")


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'derek-thomas/ScienceQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading ScienceQA...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-1028f23e353fbe(…):   0%|          | 0.00/377M [00:00<?, ?B/s]

data/validation-00000-of-00001-6c7328ff6(…):   0%|          | 0.00/126M [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

  VQA-v2: 500 items (skip=0)

✓ Test sets ready:
  ScienceQA test : 500 items
  VQA-v2 test    : 500 items


In [39]:
# CELL 9 — Evaluation Metrics

def compute_metrics(responses, ground_truths):
    """Compute TSR, SCS, HR, EMA from lists of responses and ground truths."""
    assert len(responses) == len(ground_truths), "Length mismatch"
    scores, hits, em = [], [], []
    for resp, gt in zip(responses, ground_truths):
        s = fgm_score(resp, gt)
        scores.append(s)
        hits.append(1 if s >= CFG["FGM_THRESHOLD"] else 0)
        em.append(1 if gt.lower().strip() in resp.lower() else 0)
    return {
        "Task_Success_Rate":          round(np.mean(hits),  4),
        "Semantic_Consistency_Score": round(np.mean(scores),4),
        "Hallucination_Rate":         round(1 - np.mean(hits),4),
        "Exact_Match_Accuracy":       round(np.mean(em),   4),
        "Mean_FGM_Score":             round(np.mean(scores),4),
        "n":                          len(responses),
    }

def ci95(values):
    """95% CI via t-distribution."""
    n = len(values)
    if n < 2: return (float(values[0]) if n==1 else 0, 0)
    m, se = np.mean(values), scipy_stats.sem(values)
    t = scipy_stats.t.ppf(0.975, df=n-1)
    return round(m-t*se,4), round(m+t*se,4)

print("✓ compute_metrics and ci95 ready")

✓ compute_metrics and ci95 ready


In [40]:
# CELL 10 — Training Loop (REINFORCE + optional PPO-Clip)

def run_training(model, config_name, questions, seed=42,
                 n_steps=None, use_ppo_clip=False):
    """
    REINFORCE with EMA baseline.
    If use_ppo_clip=True: applies clipped surrogate loss (true PPO).
    Returns: (summary_df, grad_df)
    """
    if n_steps is None: n_steps = CFG["PPO_STEPS"]
    set_seed(seed)

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CFG["LEARNING_RATE"], weight_decay=0.01
    )
    baseline  = 0.0
    epsilon   = CFG["PPO_EPSILON"]
    algo      = "PPO-Clip" if use_ppo_clip else "REINFORCE"

    reward_log, grad_log = [], []
    window_r, window_fgm, window_tsr = [], [], []

    print(f"\n{'='*60}")
    print(f"Training [{config_name}] [{algo}] seed={seed} steps={n_steps}")
    print(f"{'='*60}")

    for step in range(n_steps):
        q = questions[step % len(questions)]

        # Rollout
        prompt = f"USER: <image>\n{q['prompt']}\nASSISTANT:"
        inputs = processor(text=prompt, images=q["image"],
                           return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=CFG["MAX_NEW_TOKENS"],
                do_sample=True,
                temperature=CFG["TEMPERATURE"],
                pad_token_id=processor.tokenizer.eos_token_id,
            )
        new_tok  = gen_ids[0][inputs.input_ids.shape[1]:]
        response = processor.decode(new_tok, skip_special_tokens=True)

        # Reward + advantage
        fs      = fgm_score(response, q["ground_truth"])
        reward  = compute_reward(config_name, fs)
        adv     = reward - baseline
        baseline = CFG["EMA_DECAY"]*baseline + (1-CFG["EMA_DECAY"])*reward

        # Forward pass
        full_seq = torch.cat([inputs.input_ids, new_tok.unsqueeze(0)], dim=1)
        labels   = full_seq.clone()
        labels[:, :inputs.input_ids.shape[1]] = -100
        out      = model(input_ids=full_seq,
                         attention_mask=torch.ones_like(full_seq),
                         labels=labels)
        log_prob = -out.loss

        # Loss
        adv_t = torch.tensor(adv, dtype=torch.float32, device=log_prob.device)
        if use_ppo_clip:
            with torch.no_grad():
                old_out  = model(input_ids=full_seq,
                                 attention_mask=torch.ones_like(full_seq),
                                 labels=labels)
                old_lp   = -old_out.loss.detach()
            ratio   = torch.exp(log_prob - old_lp)
            clipped = torch.clamp(ratio, 1-epsilon, 1+epsilon)
            loss    = -torch.min(ratio*adv_t, clipped*adv_t)
        else:
            loss = -log_prob * adv_t

        # Backward
        optimizer.zero_grad()
        loss.backward()

        # Gradient norm (before clipping)
        raw_gn = sum(
            p.grad.data.norm(2).item()**2
            for p in model.parameters() if p.grad is not None
        ) ** 0.5
        clip_gn = torch.nn.utils.clip_grad_norm_(model.parameters(),
                                                   CFG["GRAD_CLIP"]).item()
        optimizer.step()

        # Log
        window_r.append(reward); window_fgm.append(fs)
        window_tsr.append(float(fs > CFG["TASK_THRESHOLD"]))
        grad_log.append({"step":step+1,"raw_grad_norm":raw_gn,
                         "clipped_norm":clip_gn,"reward":reward,
                         "advantage":adv,"fgm":fs})

        if (step+1) % 50 == 0:
            rm  = np.mean(window_r[-50:])
            fm  = np.mean(window_fgm[-50:])
            tsr = np.mean(window_tsr[-50:])
            gn  = np.mean([g["clipped_norm"] for g in grad_log[-50:]])
            print(f"  Step {step+1:4d}/{n_steps} | R={rm:+.3f} | "
                  f"FGM={fm:.3f} | TSR={tsr:.3f} | GradNorm={gn:.4f}")
            reward_log.append({"step":step+1,"reward_mean":rm,
                               "fgm_mean":fm,"tsr":tsr,"grad_norm":gn,
                               "config":config_name,"seed":seed,"algo":algo})

    print(f"  Done. Mean reward={np.mean(window_r):.4f}  "
          f"Final-50 mean={np.mean(window_r[-50:]):.4f}")

    summary_df = pd.DataFrame(reward_log)
    grad_df    = pd.DataFrame(grad_log)
    grad_df["config"] = config_name
    grad_df["seed"]   = seed
    grad_df["algo"]   = algo
    return summary_df, grad_df

print("✓ run_training ready")


✓ run_training ready


In [41]:
# CELL 11 — Evaluation Loop

def run_evaluation(label, model, questions, seed=42, log_interval=100):
    """Evaluate model on questions. Returns (responses, ground_truths)."""
    set_seed(seed)
    responses, gts = [], []
    print(f"Evaluating [{label}] n={len(questions)} seed={seed}")
    for i, q in enumerate(questions):
        prompt = f"USER: <image>\n{q['prompt']}\nASSISTANT:"
        inputs = processor(text=prompt, images=q["image"],
                           return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=CFG["MAX_NEW_TOKENS"],
                do_sample=True,
                temperature=CFG["EVAL_TEMPERATURE"],
                pad_token_id=processor.tokenizer.eos_token_id,
            )
        new_tok  = gen_ids[0][inputs.input_ids.shape[1]:]
        response = processor.decode(new_tok, skip_special_tokens=True)
        responses.append(response)
        gts.append(q["ground_truth"])
        if (i+1) % log_interval == 0:
            s = fgm_score(response, q["ground_truth"])
            print(f"  Q {i+1}/{len(questions)}: FGM={s:.3f} | {response[:60]}")
    return responses, gts

print("✓ run_evaluation ready")

✓ run_evaluation ready


In [ ]:
# ══════════════════════════════════════════════════════════════
# PASTE THIS CELL — run before experiment cell
# Full P100 (sm_60) compatibility patch for PyTorch 2.10+
# ══════════════════════════════════════════════════════════════
import gc, torch, os
import torch.nn.functional as F
from transformers.generation.utils import GenerationMixin

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# ── Patch 1: _prepare_special_tokens ─────────────────────────
# This method does validation checks using CUDA tensor ops that
# require sm_70+. We replace it with a CPU-safe version that
# sets the exact same attributes the rest of generate() expects.

def _cpu_safe_prepare_special_tokens(
    self, generation_config, kwargs_has_attention_mask=None, device=None
):
    """P100-safe _prepare_special_tokens: all checks run on CPU."""
    import torch as _t

    def _to_tensor(val, device):
        if val is None:
            return None
        if not isinstance(val, (list, tuple)):
            val = [val]
        # Create on CPU for safety checks, then move to device
        t_cpu = _t.tensor(val, dtype=_t.long, device="cpu")
        # Validation on CPU (no sm_70+ needed)
        if _t.is_floating_point(t_cpu) or (t_cpu < 0).any():
            return None   # invalid token — skip silently
        return t_cpu.to(device) if device is not None else t_cpu

    # Set the token tensors that generate() reads after this call
    generation_config._eos_token_tensor = _to_tensor(
        getattr(generation_config, "eos_token_id", None), device
    )
    generation_config._pad_token_tensor = _to_tensor(
        getattr(generation_config, "pad_token_id", None), device
    )
    generation_config._bos_token_tensor = _to_tensor(
        getattr(generation_config, "bos_token_id", None), device
    )
    generation_config._decoder_start_token_tensor = _to_tensor(
        getattr(generation_config, "decoder_start_token_id", None), device
    )

GenerationMixin._prepare_special_tokens = _cpu_safe_prepare_special_tokens
print("✓ _prepare_special_tokens patched (CPU-safe)")

# ── Patch 2: torch.isin CPU fallback ────────────────────────
_orig_isin = torch.isin
def _safe_isin(elements, test_elements, **kwargs):
    dev = getattr(elements, "device", "cpu")
    te  = test_elements.cpu() if hasattr(test_elements, "cpu") else test_elements
    return _orig_isin(elements.cpu(), te, **kwargs).to(dev)
torch.isin = _safe_isin
print("✓ torch.isin patched (CPU fallback)")

# ── Clear GPU memory ─────────────────────────────────────────
print("Clearing GPU memory...")
for name in list(globals().keys()):
    try:
        obj = globals()[name]
        if hasattr(obj, "parameters"):
            obj.cpu(); del globals()[name]
        elif isinstance(obj, torch.Tensor) and obj.is_cuda:
            del globals()[name]
    except: pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
free, total = torch.cuda.mem_get_info()
print(f"✓ VRAM: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total")

# ── Redefine nuke() ──────────────────────────────────────────
def nuke():
    for name in ["m","base","model","sm_reinforce","sm_ppoclip",
                 "sm_hra","vm","vanilla_model","lora_model"]:
        if name in globals():
            try:    globals()[name].cpu()
            except: pass
            try:    del globals()[name]
            except: pass
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM: {free/1024**3:.2f} GB free / {total/1024**3:.2f} GB total")

# ── Redefine model factory ───────────────────────────────────
from peft import get_peft_model, LoraConfig
from transformers import LlavaForConditionalGeneration

def _lora():
    return LoraConfig(
        r=CFG["LORA_RANK"], lora_alpha=CFG["LORA_RANK"]*2,
        lora_dropout=0.05, bias="none",
        target_modules=["q_proj","v_proj"],
        task_type="CAUSAL_LM",
    )

def make_lora_model():
    gc.collect(); torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM before load: {free/1024**3:.2f} GB / {total/1024**3:.2f} GB")
    base = LlavaForConditionalGeneration.from_pretrained(
        CFG["MODEL_ID"], device_map="auto",
        torch_dtype=torch.float16,
        cache_dir=CACHE_DIR, local_files_only=already_cached,
    )
    model = get_peft_model(base, _lora(), autocast_adapter_dtype=False)
    model.print_trainable_parameters()
    free2, _ = torch.cuda.mem_get_info()
    print(f"  VRAM after load:  {free2/1024**3:.2f} GB")
    return model

def make_vanilla_model():
    gc.collect(); torch.cuda.empty_cache()
    m = LlavaForConditionalGeneration.from_pretrained(
        CFG["MODEL_ID"], device_map="auto",
        torch_dtype=torch.float16,
        cache_dir=CACHE_DIR, local_files_only=already_cached,
    )
    free, _ = torch.cuda.mem_get_info()
    print(f"  Vanilla loaded. VRAM: {free/1024**3:.2f} GB")
    return m

print("✓ Model factory redefined")
print("\n✓ All P100 patches applied. Run experiment cell now.")


✓ _prepare_special_tokens patched (CPU-safe)
✓ torch.isin patched (CPU fallback)
Clearing GPU memory...


In [47]:
# CELL 12 — Multi-Seed Experiment (8 seeds, 3 agents, 2 benchmarks)

# Storage
results   = {
    "scienceqa": {"reinforce":[],"ppo_clip":[],"hra":[],"vanilla":[]},
    "vqa_v2":    {"reinforce":[],"ppo_clip":[],"hra":[],"vanilla":[]},
}
grad_store = {"reinforce":[],"ppo_clip":[],"hra":[]}
train_logs = {"reinforce":[],"ppo_clip":[],"hra":[]}
completed  = []

# Auto-resume
ckpts = sorted(
    [f for f in os.listdir(WORK_DIR) if f.startswith("ckpt_seed") and f.endswith(".json")],
    key=lambda x: os.path.getmtime(os.path.join(WORK_DIR,x))
)
if ckpts:
    try:
        with open(os.path.join(WORK_DIR, ckpts[-1])) as f:
            saved = json.load(f)
        completed = saved.get("completed",[])
        print(f"Resuming. Completed seeds: {completed}")
    except: pass

to_run = [s for s in CFG["SEEDS"] if s not in completed]
print(f"Seeds to run: {to_run}")

# ── Main loop ─────────────────────────────────────────────
for seed in to_run:
    sep = "-"*50
    print(f"\n{sep}\n  SEED {seed}\n{sep}")

    # Build datasets
    TRAIN_SQ = build_scienceqa("train", CFG["TRAIN_ITEMS"], seed)
    TRAIN_VQ = build_vqa(CFG["TRAIN_ITEMS"], skip=CFG["TEST_ITEMS"], seed=seed)

    for agent, cfg_name, use_clip in [
        ("reinforce", "E: Std REINFORCE", False),
        ("ppo_clip",  "E: Std REINFORCE", True),
        ("hra",       "D: Full HRA",      False),
    ]:
        nuke()
        algo = "PPO-Clip" if use_clip else "REINFORCE"
        print(f"\n  [{algo}] seed={seed}")
        m = make_lora_model()

        # Train on ScienceQA
        s_df, g_df = run_training(m, cfg_name, TRAIN_SQ,
                                   seed=seed, use_ppo_clip=use_clip)
        g_df["agent"] = agent
        grad_store[agent].append(g_df)
        train_logs[agent].append(s_df)

        # Eval ScienceQA
        r, gt = run_evaluation(f"{agent}-SciQA", m, SCIQA_TEST, seed=42)
        met   = compute_metrics(r, gt)
        met.update({"seed":seed,"agent":agent,"benchmark":"scienceqa","algo":algo})
        results["scienceqa"][agent].append(met)

        # Eval VQA-v2
        if VQA_TEST:
            r_v, gt_v = run_evaluation(f"{agent}-VQA", m, VQA_TEST, seed=42)
            met_v     = compute_metrics(r_v, gt_v)
            met_v.update({"seed":seed,"agent":agent,"benchmark":"vqa_v2","algo":algo})
            results["vqa_v2"][agent].append(met_v)

        nuke()

    # Vanilla (no training)
    nuke()
    print(f"\n  [Vanilla] seed={seed}")
    vm = make_vanilla_model()
    for bm, testset in [("scienceqa",SCIQA_TEST),("vqa_v2",VQA_TEST)]:
        if testset:
            r,gt = run_evaluation(f"Vanilla-{bm}", vm, testset, seed=42)
            met  = compute_metrics(r,gt)
            met.update({"seed":seed,"agent":"vanilla","benchmark":bm,"algo":"none"})
            results[bm]["vanilla"].append(met)
    nuke()

    # Checkpoint
    completed.append(seed)
    ckpt_path = os.path.join(WORK_DIR, f"ckpt_seed{seed}.json")
    with open(ckpt_path,"w") as f:
        json.dump({"completed":completed,
                   "results":{bm:{ag:[str(r) for r in rl]
                               for ag,rl in agents.items()}
                              for bm,agents in results.items()}}, f)
    print(f"  Checkpoint: {ckpt_path}")

    del TRAIN_SQ, TRAIN_VQ
    gc.collect()

print("\n✅ All seeds complete.")

Seeds to run: [42, 123, 7, 1, 2, 3, 4, 5]

--------------------------------------------------
  SEED 42
--------------------------------------------------
  ScienceQA train: 400 items (seed=42)


Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/68 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/143 [00:00<?, ?it/s]

  VQA-v2: 400 items (skip=500)
  VRAM: 15.61 GB free / 15.89 GB total

  [REINFORCE] seed=42
  VRAM before load: 15.61 GB / 15.89 GB


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

trainable params: 9,961,472 || all params: 7,073,388,544 || trainable%: 0.1408
  VRAM after load:  2.44 GB

Training [E: Std REINFORCE] [REINFORCE] seed=42 steps=400


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# CELL 13 — Statistical Summary

def summarise(rlist, metric="Task_Success_Rate"):
    vals = [r[metric] for r in rlist if metric in r]
    if len(vals) < 2: return {}
    n  = len(vals)
    m  = np.mean(vals)
    s  = np.std(vals, ddof=1)
    lo, hi = ci95(vals)
    return {"mean":m,"std":s,"ci_lo":lo,"ci_hi":hi,
            "n":n,"vals":vals,"in_01": 0<=lo and hi<=1}

METRICS = ["Task_Success_Rate","Semantic_Consistency_Score","Hallucination_Rate"]
AGENTS  = ["vanilla","reinforce","ppo_clip","hra"]

for bm in ["scienceqa","vqa_v2"]:
    print(f"\n{'='*70}")
    print(f"BENCHMARK: {bm.upper()}  |  n={len(CFG['SEEDS'])} seeds")
    print(f"{'='*70}")
    for metric in METRICS:
        print(f"\n  {metric}")
        print(f"  {'Agent':<14} {'Mean':>7} {'SD':>7} {'95% CI':>22} {'Valid CI'}")
        print(f"  {'-'*65}")
        sums = {}
        for ag in AGENTS:
            rlist = results[bm].get(ag,[])
            if not rlist: continue
            s = summarise(rlist, metric)
            if not s: continue
            sums[ag] = s
            v = "✓" if s["in_01"] else "⚠ outside [0,1]"
            print(f"  {ag:<14} {s['mean']:>7.4f} {s['std']:>7.4f} "
                  f"  [{s['ci_lo']:+.4f},{s['ci_hi']:+.4f}]   {v}")

        # REINFORCE vs HRA
        if "reinforce" in sums and "hra" in sums:
            rv = sums["reinforce"]["vals"]
            hv = sums["hra"]["vals"]
            mn = min(len(rv),len(hv))
            if mn >= 2:
                d = (np.mean(rv[:mn])-np.mean(hv[:mn])) / (
                    np.sqrt(((mn-1)*np.std(rv[:mn],ddof=1)**2+
                             (mn-1)*np.std(hv[:mn],ddof=1)**2)/(2*mn-2)) or 1)
                t,p = scipy_stats.ttest_rel(rv[:mn],hv[:mn])
                eff = "large" if abs(d)>=0.8 else "medium" if abs(d)>=0.5 else "small"
                sig = "p<.05 ✓" if p<0.05 else f"p={p:.3f}"
                print(f"\n  REINFORCE vs HRA: d={d:.3f} ({eff})  {sig}")

print("\n✅ Statistics done.")
